# 🔑 Solusi — Prediksi Respon Kampanye Marketing (Bab 3)> Buka HANYA setelah mencoba sendiri. Fokus pada **cara membaca angka**,> bukan sekadar mencocokkan hasil.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))   # biar `ml_sederhana` bisa diimpor dari notebook

import matplotlib.pyplot as plt
import numpy as np

from ml_sederhana.clustering import kmeans, standardize
from ml_sederhana.data import muat_semua
from ml_sederhana.metrik import precision_recall_f1, roc_auc
from ml_sederhana.model import LogisticRegressionGD

X_train, y_train, X_val, y_val, X_test, y_test, NAMA = muat_semua()
print('train/val/test:', len(y_train), len(y_val), len(y_test),
      '| positif:', y_train.mean().round(3), y_val.mean().round(3), y_test.mean().round(3))


## Bagian 1 — Baseline vs Model

In [ ]:
y_bodoh = np.zeros_like(y_val)
print(f'Accuracy model bodoh di val: {(y_bodoh == y_val).mean():.4f}')

model = LogisticRegressionGD(lr=0.1, n_iter=2000).fit(X_train, y_train)
p_val = model.predict_proba(X_val)
pred_val = model.predict(X_val)

p, r, f1 = precision_recall_f1(y_val, pred_val)
print(f'Model      : accuracy={(pred_val == y_val).mean():.4f}  precision={p:.3f}  recall={r:.3f}  f1={f1:.3f}')
print('Model bodoh: precision=0.0 (tidak pernah menebak positif)  recall=0.0')
print('-> Selisih accuracy tipis, tapi kemampuan MENEMUKAN pembeli hanya dimiliki model.')


## Bagian 2 — Kurva Loss

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(model.loss_history)
plt.xlabel('iterasi'); plt.ylabel('BCE (train)'); plt.title('Kurva Loss')
plt.grid(alpha=0.3); plt.show()
print(f'loss awal {model.loss_history[0]:.4f} (≈ ln 2 = 0.6931, tebakan 50:50) -> akhir {model.loss_history[-1]:.4f}')


## Bagian 3 — Eksperimen Threshold

In [ ]:
rows = []
for th in np.round(np.arange(0.05, 0.96, 0.05), 2):
    pred = (p_val >= th).astype(int)
    pr, rc, _ = precision_recall_f1(y_val, pred)
    rows.append((th, pr, rc))

print(f"{'th':>5} | {'precision':>9} | {'recall':>7}")
print('-' * 30)
for th, pr, rc in rows:
    print(f'{th:>5.2f} | {pr:>9.3f} | {rc:>7.3f}')

# FN (lewatkan pembeli) jauh lebih mahal -> pilih recall tinggi dengan precision masih layak.
# Dari tabel: th ≈ 0.30-0.40 memberi recall ≥ 0.7 dengan precision ≈ 0.4-0.5.


**Bacaan hasil:** di th=0.5 model mendapat precision 0.65 / recall 0.48 —hampir setengah pembeli terlewat. Menurunkan threshold ke ~0.35 menukarsedikit precision dengan recall yang jauh lebih tinggi; untuk kampanye murahper-kontak, itu pertukaran yang masuk akal.

## Bagian 4 — ROC-AUC

In [ ]:
auc_val = roc_auc(y_val, p_val)
print(f'AUC val: {auc_val:.4f}')
print(f'-> Dua pelanggan acak (satu pembeli, satu bukan): model mengurutkan benar ~{auc_val:.0%} waktu.')


## Bagian 5 — Segmen Pelanggan

In [ ]:
X_all = np.vstack([X_train, X_val, X_test])
y_all = np.concatenate([y_train, y_val, y_test])

Z = standardize(X_all)
labels, centroids, inertia = kmeans(Z, k=2, seed=0)
print(f'inertia: {inertia:.1f}\n')
for k in range(2):
    anggota = y_all[labels == k]
    print(f'klaster {k}: {len(anggota):>4} orang, positive-rate {anggota.mean():.4f}')

print('\ncentroid (fitur 0-2, terskala):')
print(centroids[:, :3].round(2))
print('fitur paling membedakan klaster:', NAMA[int(np.argmax(np.abs(centroids[0] - centroids[1])))])


**Bacaan hasil:** klaster terpisah terutama oleh `jam_pakai_per_minggu`(fitur bimodal yang paling membedakan kelas). Segmen dengan positive-ratetinggi (~17%) adalah target promo; segmen rendah (~4%) nyaris tidak worthdikontak — padahal k-means tidak pernah melihat label sedikit pun.

## Bagian 6 — Test Set, Sekali

In [ ]:
TH_PILIHAN = 0.35   # dipilih di validation (Bagian 3), BUKAN dari test

p_test = model.predict_proba(X_test)
pred_test = (p_test >= TH_PILIHAN).astype(int)
print(f'accuracy test @th={TH_PILIHAN}: {(pred_test == y_test).mean():.4f}')
print(f'precision/recall test : {precision_recall_f1(y_test, pred_test)}')
print(f'AUC test              : {roc_auc(y_test, p_test):.4f}')
print(f'AUC val               : {auc_val:.4f}  (bandingkan: beda tipis = sehat)')


## 📝 Contoh Jawaban Analisis (ringkas)1. Accuracy menipu karena 89% kelas negatif: menebak 'tidak semua' sudah 0.89.   Precision/recall memperlihatkan kemampuan menemukan pembeli — model bodoh 0/0, model kita 0.65/0.48.2. Threshold ~0.35: FN (kehilangan pembeli) jauh lebih mahal daripada FP (promo mubazir),   jadi bayar precision sedikit demi recall jauh lebih tinggi.3. AUC menilai ranking skor (invariant threshold); F1 menilai keputusan biner   pada satu titik threshold — geser threshold, keputusan berubah, ranking tidak.4. Segmen positive-rate ~17% vs ~4%: kontak segmen tinggi → ~4x lipat kemungkinan konversi   per promo; justifikasi pakai positive-rate × ukuran segmen.5. Val vs test beda ~1-2 poin persen: wajar (kebetulan sampling). Bahaya bila test   dipakai berulang untuk memilih model → estimasi produksi jadi terlalu optimis.